# 6.7 层级协同：三无人机工业巡检

> **AirSim 配置**：本节使用 `7-settings.json`（3架无人机：Drone1、Drone2、Drone3）。运行下方代码自动切换配置并重启 AirSim。

本节实现一个更贴近真实场景的多无人机协同模式——**层级指挥**：

```
                    Drone3（监控指挥）
                   ┌──── 高空悬停 ────┐
                   │  获取目标位置     │
                   │  LLM制定计划      │
                   │  收集汇报+总结    │
                   └──┬──────────┬──┘
                      │          │
              分配任务↓          ↓分配任务
                      │          │
            Drone1（巡检员A）  Drone2（巡检员B）
            → 电塔群            → 变压器设备
            → 近距离拍照        → 近距离拍照
            → 汇报状态          → 汇报状态
```

### 场景

当前 AirSim 工业巡检场景中包含：
- **电塔群** — 场景东部区域（Obj3d66-510534 系列）
- **变压器/电气设备** — 场景中部（dfz 系列）
- **施工人员** — 场景西部（SkeletalMeshActor 系列）

我们通过 AirSim API **自动扫描**这些设施的真实坐标，而非硬编码。

## Step 0：切换到3架无人机配置

本节需要3架无人机，先切换 settings.json 并重启 AirSim。

In [6]:
# 切换到3架无人机配置并重启 AirSim
from airsim_tools import apply_settings
apply_settings("7-settings.json")

已复制 7-settings.json → /root/Documents/AirSim/settings.json
正在重启 AirSim，等待 10 秒...
AirSim 已重启


## Step 1：环境准备

连接 AirSim，定义辅助函数。

In [7]:
import sys
sys.path.append('../external-libraries')

import json
import math
import numpy as np
from PIL import Image
import airsim
from airsim_tools import call_llm, takeoff, fly_to, get_state

# 连接 AirSim
client = airsim.MultirotorClient()
client.confirmConnection()
print(f"连接成功！无人机列表: {client.listVehicles()}")


def get_inspection_targets(client):
    """通过 AirSim API 获取场景中的巡检目标位置。"""
    targets = {}
    objects = client.simListSceneObjects()

    # 1. 电塔群（Obj3d66-510534 开头）
    tower_positions = []
    for obj in objects:
        if obj.startswith('Obj3d66-510534-'):
            pose = client.simGetObjectPose(obj)
            if not math.isnan(pose.position.x_val):
                tower_positions.append((pose.position.x_val, pose.position.y_val, pose.position.z_val))
    if tower_positions:
        avg_x = round(sum(p[0] for p in tower_positions) / len(tower_positions), 1)
        avg_y = round(sum(p[1] for p in tower_positions) / len(tower_positions), 1)
        targets['电塔群'] = {
            'name': f'电塔({len(tower_positions)}座)',
            'x': avg_x, 'y': avg_y, 'z': -15.0
        }

    # 2. 变压器/电气设备（dfz 开头）
    dfz_positions = []
    for obj in objects:
        if obj.startswith('dfz'):
            pose = client.simGetObjectPose(obj)
            if not math.isnan(pose.position.x_val):
                dfz_positions.append((pose.position.x_val, pose.position.y_val, pose.position.z_val))
    if dfz_positions:
        avg_x = round(sum(p[0] for p in dfz_positions) / len(dfz_positions), 1)
        avg_y = round(sum(p[1] for p in dfz_positions) / len(dfz_positions), 1)
        targets['变压器设备'] = {
            'name': f'变压器区域({len(dfz_positions)}台)',
            'x': avg_x, 'y': avg_y, 'z': -10.0
        }

    # 3. 工人/人员（SkeletalMeshActor 开头）
    worker_positions = []
    for obj in objects:
        if obj.startswith('SkeletalMeshActor'):
            pose = client.simGetObjectPose(obj)
            if not math.isnan(pose.position.x_val):
                worker_positions.append((pose.position.x_val, pose.position.y_val, pose.position.z_val))
    if worker_positions:
        avg_x = round(sum(p[0] for p in worker_positions) / len(worker_positions), 1)
        avg_y = round(sum(p[1] for p in worker_positions) / len(worker_positions), 1)
        targets['施工人员区'] = {
            'name': f'工人({len(worker_positions)}人)',
            'x': avg_x, 'y': avg_y, 'z': -8.0
        }

    return targets


def capture_image(client, drone_id, filename):
    """从指定无人机拍摄照片并保存。"""
    responses = client.simGetImages([
        airsim.ImageRequest("0", airsim.ImageType.Scene, False, False)
    ], vehicle_name=drone_id)

    if responses and responses[0].width > 0:
        img = np.frombuffer(responses[0].image_data_uint8, dtype=np.uint8)
        img = img.reshape(responses[0].height, responses[0].width, 3)
        Image.fromarray(img[:, :, ::-1]).save(filename)
        print(f"[{drone_id}] 照片已保存: {filename}")
        return filename
    else:
        print(f"[{drone_id}] 拍照失败")
        return None


# 获取巡检目标
targets = get_inspection_targets(client)
print(f"\n发现 {len(targets)} 个巡检目标:")
for name, info in targets.items():
    print(f"  {name}: {info['name']} @ ({info['x']}, {info['y']}, {info['z']})")

Connected!
Client Ver:1 (Min Req: 1), Server Ver:1 (Min Req: 1)

连接成功！无人机列表: ['Drone1', 'Drone2', 'Drone3']

发现 3 个巡检目标:
  电塔群: 电塔(83座) @ (22.4, 92.6, -15.0)
  变压器设备: 变压器区域(21台) @ (-32.5, 86.7, -10.0)
  施工人员区: 工人(10人) @ (-50.6, 33.2, -8.0)


In [8]:
# Drone3 起飞到高空中心位置
MONITOR_DRONE = "Drone3"
MONITOR_POS = (90, 30, -40)  # 高空位置，俯瞰整个场景

takeoff(client, MONITOR_DRONE)
fly_to(client, MONITOR_DRONE, *MONITOR_POS)

# 拍摄全局视图
capture_image(client, MONITOR_DRONE, "nb7_global_view.png")
print(f"\n[{MONITOR_DRONE}] 监控无人机已就位: {get_state(client, MONITOR_DRONE)}")

[Drone3] 起飞完成
[Drone3] 已到达 (90, 30, -40)
[Drone3] 照片已保存: nb7_global_view.png

[Drone3] 监控无人机已就位: {'x': 91.5, 'y': 30.51, 'z': -40.32}


## Step 3：LLM 制定巡检计划

监控无人机将场景目标信息发送给 LLM，由 LLM 制定具体的巡检任务分配。

In [9]:
# 构建目标信息
target_info = json.dumps(targets, ensure_ascii=False, indent=2)

plan_prompt = f"""你是无人机巡检任务指挥官。现在有以下巡检目标：

{target_info}

可用无人机：Drone1（巡检员A）、Drone2（巡检员B）。
NED坐标系：z为负表示高度，巡检高度建议 z=-15（15米高）。

请为每架无人机分配一个巡检目标，输出JSON格式：
{{
  "tasks": [
    {{"drone_id": "Drone1", "target": "目标名", "x": 数值, "y": 数值, "z": -15, "description": "任务描述"}},
    {{"drone_id": "Drone2", "target": "目标名", "x": 数值, "y": 数值, "z": -15, "description": "任务描述"}}
  ]
}}

只输出JSON，不要其他内容。"""

plan_response = call_llm(plan_prompt, system="你是无人机巡检指挥官，请严格按JSON格式输出。")
print("LLM 巡检计划:")
print(plan_response)

# 解析计划
# 提取JSON（处理可能的markdown代码块包裹）
plan_text = plan_response.strip()
if '```' in plan_text:
    plan_text = plan_text.split('```')[1]
    if plan_text.startswith('json'):
        plan_text = plan_text[4:]
    plan_text = plan_text.strip()

plan = json.loads(plan_text)
print(f"\n解析成功，共 {len(plan['tasks'])} 个任务")

LLM 巡检计划:
{
  "tasks": [
    {"drone_id": "Drone1", "target": "电塔(83座)", "x": 22.4, "y": 92.6, "z": -15, "description": "巡检指定区域内83座电塔，排查电塔结构损伤、输电线路松动等异常隐患"},
    {"drone_id": "Drone2", "target": "变压器区域(21台)", "x": -32.5, "y": 86.7, "z": -15, "description": "巡检指定区域内21台变压器设备，核查设备运行状态，排查过热、漏油等故障问题"}
  ]
}

解析成功，共 2 个任务


## Step 4：巡检无人机执行任务

Drone1 和 Drone2 按照指挥官的计划，分别飞往各自的巡检目标，拍照并汇报。

In [10]:
# 执行巡检任务
reports = []

for task in plan['tasks']:
    drone_id = task['drone_id']
    target_name = task['target']
    x, y, z = task['x'], task['y'], task['z']
    
    print(f"\n{'='*50}")
    print(f"[{drone_id}] 执行任务: 巡检{target_name}")
    print(f"{'='*50}")
    
    # 1. 起飞
    takeoff(client, drone_id)
    
    # 2. 飞往目标
    fly_to(client, drone_id, x, y, z)
    
    # 3. 拍照
    photo = capture_image(client, drone_id, f"nb7_{drone_id}_{target_name}.png")
    
    # 4. 获取实际位置
    pos = get_state(client, drone_id)
    
    # 5. 记录报告
    report = {
        'drone_id': drone_id,
        'target': target_name,
        'target_coords': f'({x}, {y}, {z})',
        'actual_position': pos,
        'photo': photo,
        'status': '到达目标，拍照完成'
    }
    reports.append(report)
    print(f"[{drone_id}] 汇报: 已到达{target_name}，实际位置 {pos}")

print(f"\n所有巡检任务执行完毕，共收到 {len(reports)} 份汇报")


[Drone1] 执行任务: 巡检电塔(83座)
[Drone1] 起飞完成
[Drone1] 已到达 (22.4, 92.6, -15)
[Drone1] 照片已保存: nb7_Drone1_电塔(83座).png
[Drone1] 汇报: 已到达电塔(83座)，实际位置 {'x': 22.77, 'y': 94.09, 'z': -14.96}

[Drone2] 执行任务: 巡检变压器区域(21台)
[Drone2] 起飞完成
[Drone2] 已到达 (-32.5, 86.7, -15)
[Drone2] 照片已保存: nb7_Drone2_变压器区域(21台).png
[Drone2] 汇报: 已到达变压器区域(21台)，实际位置 {'x': -33.03, 'y': 88.08, 'z': -14.93}

所有巡检任务执行完毕，共收到 2 份汇报


## Step 5：监控无人机汇总报告

Drone3 收集所有巡检无人机的汇报，交给 LLM 生成最终巡检报告。

In [5]:
# 监控无人机拍摄最终全局视图
capture_image(client, MONITOR_DRONE, "nb7_final_overview.png")

# 构建汇报信息
report_info = json.dumps(reports, ensure_ascii=False, indent=2)

summary_prompt = f"""你是无人机巡检指挥官。以下是两架巡检无人机的任务执行汇报：

{report_info}

请生成一份简洁的巡检总结报告，包含：
1. 巡检概况（几架无人机、几个目标）
2. 各目标巡检结果
3. 总体评估和建议

用中文输出，200字以内。"""

summary = call_llm(summary_prompt, system="你是无人机巡检指挥官，请输出简洁的巡检报告。")

print("="*60)
print("         巡检总结报告（LLM 生成）")
print("="*60)
print(summary)
print("="*60)

[Drone3] 照片已保存: nb7_final_overview.png
         巡检总结报告（LLM 生成）
# 无人机巡检总结报告
1. 巡检概况：本次出动2架无人机，共巡检2个目标（83座电塔、21台变压器区域），全部完成抵点拍摄。
2. 各目标结果：Drone1完成电塔群巡检，存照`nb7_Drone1_电塔(83座).png`；Drone2完成变压器区域巡检，存照`nb7_Drone2_变压器区域(21台).png`，双机就位偏差均符合要求。
3. 总体评估：本次任务全部顺利完成，影像资料齐全，建议尽快开展拍摄照片的设备缺陷排查。


## 总结：三种协同模式对比

| 对比项 | 中心化（笔记本3） | 分布式（笔记本4） | 层级指挥（本节） |
|--------|-------------------|-------------------|------------------|
| 决策方式 | 1个LLM统一规划 | 每架无人机各自决策 | 指挥官规划 + 执行者汇报 |
| 通信模式 | 指挥官→工作者 | 消息板（双向） | 指挥官↔巡检员（双向层级） |
| 角色分工 | 无明确分工 | 平等协商 | 监控/巡检明确分层 |
| 场景感知 | 硬编码坐标 | 共享状态 | API获取真实目标 + 拍照确认 |
| 适用场景 | 简单任务 | 动态环境 | 工业巡检、搜救等需要层级管理的场景 |

层级指挥模式的核心优势：
- **监控无人机**在高空拥有全局视野，适合做决策
- **巡检无人机**专注执行，近距离获取详细信息
- 信息**自下而上汇报**，决策**自上而下分配**，符合真实指挥链